In [2]:

#%pip install pandas numpy scikit-learn xgboost lightgbm tensorflow matplotlib catboost --upgrade
import akshare as ak
import pandas as pd
import numpy as np
from datetime import datetime,timedelta
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from catboost import CatBoostRegressor
import os
os.environ['LIGHTGBM_VERBOSE'] = '-1'
#from tensorflow.keras.models import Sequential
#from tensorflow.keras.layers import LSTM, Dense, Dropout
#from tensorflow.keras.optimizers import Adam

# 获取过去10年的ETF历史数据
def fetch_etf_data(symbol):
    end_date = datetime.today().strftime("%Y%m%d")
    start_date = (datetime.today() - timedelta(days=3650)).strftime("%Y%m%d")

    etf_df = ak.fund_etf_hist_em(symbol=symbol,
                                 period="daily",
                                 start_date=start_date,
                                 end_date=end_date,
                                 adjust="")
    
    print("原始ETF数据列名如下：")  # 调试信息
    print(etf_df.columns.tolist())  # 查看真实列名
    
    # 根据实际列名修改下面的映射
    column_mapping = {
        "日期": "日期",
        "开盘": "开盘",
        "最高": "最高",
        "最低": "最低",
        "收盘": "收盘",
        "成交量": "成交量"
    }

    selected_columns = {k: v for k, v in column_mapping.items() if k in etf_df.columns}
    etf_df = etf_df[selected_columns.keys()].rename(columns=selected_columns)

    etf_df['日期'] = pd.to_datetime(etf_df['日期'])
    return etf_df.sort_values('日期').reset_index(drop=True)

# 特征工程函数
def prepare_features(df):
    df = df.copy()
    df['日期'] = pd.to_datetime(df['日期'])
    df.set_index('日期', inplace=True)

    # 添加滞后特征
    for lag in range(1, 6):
        df[f'收盘_滞后{lag}'] = df['收盘'].shift(lag)
        df[f'最高_滞后{lag}'] = df['最高'].shift(lag)
        df[f'最低_滞后{lag}'] = df['最低'].shift(lag)

    # 技术指标
    df['SMA_5'] = df['收盘'].rolling(window=5).mean()
    df['Volatility'] = df['最高'] - df['最低']

    df.dropna(inplace=True)

    # 特征列
    feature_cols = [col for col in df.columns if col not in ['收盘', '最高', '最低']]
    X = df[feature_cols]
    y_high = df['最高']
    y_low = df['最低']

    return X, y_high, y_low

# 模型训练函数
def train_models(X, y_high, y_low):
    models_high = {}
    models_low = {}

    # Random Forest
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X, y_high)
    scaler = StandardScaler().fit(X)
    models_high['RandomForest'] = (model, scaler)

    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X, y_low)
    scaler = StandardScaler().fit(X)
    models_low['RandomForest'] = (model, scaler)

    # LightGBM
    model = lgb.LGBMRegressor(n_estimators=100)
    model.fit(X, y_high)
    scaler = StandardScaler().fit(X)
    models_high['LightGBM'] = (model, scaler)

    model = lgb.LGBMRegressor(n_estimators=100)
    model.fit(X, y_low)
    scaler = StandardScaler().fit(X)
    models_low['LightGBM'] = (model, scaler)

    # XGBoost
    model = xgb.XGBRegressor(n_estimators=100, use_label_encoder=False)
    model.fit(X, y_high)
    scaler = StandardScaler().fit(X)
    models_high['XGBoost'] = (model, scaler)

    model = xgb.XGBRegressor(n_estimators=100, use_label_encoder=False)
    model.fit(X, y_low)
    scaler = StandardScaler().fit(X)
    models_low['XGBoost'] = (model, scaler)

    # CatBoost
    model = CatBoostRegressor(iterations=100, verbose=0)
    model.fit(X, y_high)
    scaler = StandardScaler().fit(X)
    models_high['CatBoost'] = (model, scaler)

    model = CatBoostRegressor(iterations=100, verbose=0)
    model.fit(X, y_low)
    scaler = StandardScaler().fit(X)
    models_low['CatBoost'] = (model, scaler)

    # Hist Gradient Boosting
    model = HistGradientBoostingRegressor(max_iter=100)
    model.fit(X, y_high)
    scaler = StandardScaler().fit(X)
    models_high['HistGB'] = (model, scaler)

    model = HistGradientBoostingRegressor(max_iter=100)
    model.fit(X, y_low)
    scaler = StandardScaler().fit(X)
    models_low['HistGB'] = (model, scaler)

    # Extra Trees
    model = ExtraTreesRegressor(n_estimators=100, random_state=42)
    model.fit(X, y_high)
    scaler = StandardScaler().fit(X)
    models_high['ExtraTrees'] = (model, scaler)

    model = ExtraTreesRegressor(n_estimators=100, random_state=42)
    model.fit(X, y_low)
    scaler = StandardScaler().fit(X)
    models_low['ExtraTrees'] = (model, scaler)

    # Gradient Boosting
    model = GradientBoostingRegressor(n_estimators=100, random_state=42)
    model.fit(X, y_high)
    scaler = StandardScaler().fit(X)
    models_high['GradientBoosting'] = (model, scaler)

    model = GradientBoostingRegressor(n_estimators=100, random_state=42)
    model.fit(X, y_low)
    scaler = StandardScaler().fit(X)
    models_low['GradientBoosting'] = (model, scaler)

    # AdaBoost
    model = AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=3), n_estimators=50)
    model.fit(X, y_high)
    scaler = StandardScaler().fit(X)
    models_high['AdaBoost'] = (model, scaler)

    model = AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=3), n_estimators=50)
    model.fit(X, y_low)
    scaler = StandardScaler().fit(X)
    models_low['AdaBoost'] = (model, scaler)

    # KNN
    model = KNeighborsRegressor(n_neighbors=5)
    model.fit(X, y_high)
    scaler = StandardScaler().fit(X)
    models_high['KNN'] = (model, scaler)

    model = KNeighborsRegressor(n_neighbors=5)
    model.fit(X, y_low)
    scaler = StandardScaler().fit(X)
    models_low['KNN'] = (model, scaler)

    # SVM
    model = SVR(kernel='rbf', C=1.0, epsilon=0.1)
    model.fit(X, y_high)
    scaler = StandardScaler().fit(X)
    models_high['SVM'] = (model, scaler)

    model = SVR(kernel='rbf', C=1.0, epsilon=0.1)
    model.fit(X, y_low)
    scaler = StandardScaler().fit(X)
    models_low['SVM'] = (model, scaler)

    return models_high, models_low

def compute_adjusted_intersection(intervals, exclude_max_high=True, exclude_min_low=True):
    if exclude_max_high:
        max_upper = max([x[1] for x in intervals])
        intervals = [x for x in intervals if x[1] != max_upper]
    
    if exclude_min_low:
        min_lower = min([x[0] for x in intervals])
        intervals = [x for x in intervals if x[0] != min_lower]

    if not intervals:
        return None

    lower_bounds = [x[0] for x in intervals]
    upper_bounds = [x[1] for x in intervals]
    intersection_lower = max(lower_bounds)
    intersection_upper = min(upper_bounds)

    return (intersection_lower, intersection_upper) if intersection_lower < intersection_upper else None

import numpy as np
from scipy.signal import find_peaks

def find_most_likely_interval(intervals, precision=0.0001, show_all=False):
    """
    找出被最多模型覆盖的价格区间。
    
    参数:
        intervals: list of tuples, 每个元素是一个 (lower, upper) 区间
        precision: 浮点精度（用于划分网格）
        show_all: 是否显示所有峰值区间（True），或只返回最大覆盖的那个（False）

    返回:
        最可能区间列表，按覆盖模型数量排序
    """
    if not intervals:
        return []

    # 确定整个范围
    all_bounds = [b for interval in intervals for b in interval]
    min_val, max_val = min(all_bounds), max(all_bounds)

    # 构建网格
    grid = np.arange(min_val, max_val + precision, precision)

    # 初始化覆盖计数
    coverage = np.zeros_like(grid)

    # 统计每个位置被多少个区间覆盖
    for lower, upper in intervals:
        coverage += (grid >= lower) & (grid <= upper)

    # 查找峰值和对应位置
    peaks, _ = find_peaks(coverage)
    peak_values = coverage[peaks]
    peak_positions = grid[peaks]

    # 找出连续覆盖区间
    covered_intervals = []
    current_start = None

    for i in range(len(grid)):
        if coverage[i] > 0:
            if current_start is None:
                current_start = grid[i]
        else:
            if current_start is not None:
                covered_intervals.append((current_start, grid[i]))
                current_start = None
    if current_start is not None:
        covered_intervals.append((current_start, grid[-1]))

    # 统计每个连续区间内的最大覆盖数
    scored_intervals = []
    for start, end in covered_intervals:
        mask = (grid >= start) & (grid <= end)
        max_coverage = coverage[mask].max()
        scored_intervals.append((start, end, max_coverage))

    # 按照覆盖模型数排序
    scored_intervals.sort(key=lambda x: -x[2])

    if not show_all:
        # 只返回覆盖最多的那个区间
        return [(scored_intervals[0][0], scored_intervals[0][1])] if scored_intervals else []

    return [(s, e) for s, e, _ in scored_intervals]

def filter_outliers(predictions, threshold=0.1):
    """
    过滤掉相对于均值波动超过 threshold 的预测值。
    
    参数:
        predictions: list of float，预测值列表
        threshold: float，默认 0.1 表示 ±10%
        
    返回:
        filtered_preds: list of float，过滤后的预测值
    """
    if not predictions:
        return []

    #mean_pred = np.mean(predictions)
    end_date = datetime.today().strftime("%Y%m%d")
    start_date = (datetime.today() - timedelta(days=5)).strftime("%Y%m%d")
    etf_df = ak.fund_etf_hist_em(symbol=symbol,
                                 period="daily",
                                 start_date=start_date,
                                 end_date=end_date,
                                 adjust="")
    mean_pred = etf_df['收盘'].iloc[-1]
    filtered_preds = [
        p for p in predictions
        if abs(p - mean_pred) / mean_pred <= threshold
    ]
    return filtered_preds

def predict_future(models_high, models_low, last_row, mc_samples=100, noise_level=0.01, outlier_threshold=0.1):
    high_preds = {name: [] for name in models_high}
    low_preds = {name: [] for name in models_low}

    # 高价预测（特征扰动 + 多次预测）
    for name, (model, scaler) in models_high.items():
        base_data = scaler.transform([last_row])
        raw_base_pred = model.predict(base_data)[0]  # 原始预测值用于判断是否为异常值
        preds = []
        for _ in range(mc_samples):
            perturbed_data = base_data + np.random.normal(0, noise_level, size=base_data.shape)
            pred = model.predict(perturbed_data)[0]
            if abs(pred - raw_base_pred) / raw_base_pred <= outlier_threshold:
                preds.append(pred)
        high_preds[name] = preds  # 已过滤异常值

    # 低价预测（特征扰动 + 多次预测）
    for name, (model, scaler) in models_low.items():
        base_data = scaler.transform([last_row])
        raw_base_pred = model.predict(base_data)[0]
        preds = []
        for _ in range(mc_samples):
            perturbed_data = base_data + np.random.normal(0, noise_level, size=base_data.shape)
            pred = model.predict(perturbed_data)[0]
            if abs(pred - raw_base_pred) / raw_base_pred <= outlier_threshold:
                preds.append(pred)
        low_preds[name] = preds  # 已过滤异常值

    # 收集结果
    result = []
    all_model_avg_midpoints = []

    print("\n各模型预测值与置信区间：")
    for name in models_high.keys():
        h_preds = high_preds[name]
        l_preds = low_preds[name]

        if not h_preds or not l_preds:
            print(f"{name}：数据为空，跳过此模型。")
            continue

        h_avg = np.mean(h_preds)
        h_upper = np.percentile(h_preds, 97.5)
        h_lower = np.percentile(h_preds, 2.5)

        l_avg = np.mean(l_preds)
        l_upper = np.percentile(l_preds, 97.5)
        l_lower = np.percentile(l_preds, 2.5)

        # 模型中位预测值
        model_midpoint = (h_avg + l_avg) / 2
        all_model_avg_midpoints.append(model_midpoint)

        result.append({
            '模型': name,
            '预测最高价': h_avg,
            '上界': h_upper,
            '下界': h_lower,
            '预测最低价': l_avg,
            '最低价下界': l_lower,
            '最低价上界': l_upper,
            '中位预测值': model_midpoint
        })

        print(f"{name}:")
        print(f"  预测区间: {l_avg:.6f} ~ {h_avg:.6f} 中位预测值: {model_midpoint:.6f}\n")

    return pd.DataFrame(result)

# 主函数入口
def main(symbol):
    # 示例：模拟读取ETF数据
    df = fetch_etf_data(symbol)

    # 特征工程
    X, y_high, y_low = prepare_features(df)

    # 训练模型
    models_high, models_low = train_models(X, y_high, y_low)

    # 获取最新样本
    last_row = X.iloc[-1].copy()

    # 预测明天的最高价和最低价
    predict_future(models_high, models_low, last_row, mc_samples=100)


if __name__ == "__main__":
    symbol = input("请输入ETF代码（例如：513500）：").strip()
    main(symbol)

原始ETF数据列名如下：
['日期', '开盘', '收盘', '最高', '最低', '成交量', '成交额', '振幅', '涨跌幅', '涨跌额', '换手率']
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000158 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 626
[LightGBM] [Info] Number of data points in the train set: 103, number of used features: 19
[LightGBM] [Info] Start training from score 1.081563
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positiv

C:\Users\15601\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\training.py:183: UserWarning: [11:47:59] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\15601\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\training.py:183: UserWarning: [11:47:59] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\15601\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Us


各模型预测值与置信区间：
RandomForest:
  预测区间: 1.175420 ~ 1.186120 中位预测值: 1.180770

LightGBM:
  预测区间: 1.154753 ~ 1.165397 中位预测值: 1.160075

XGBoost:
  预测区间: 1.178518 ~ 1.186066 中位预测值: 1.182292

CatBoost:
  预测区间: 1.160526 ~ 1.170990 中位预测值: 1.165758

HistGB:
  预测区间: 1.154519 ~ 1.164728 中位预测值: 1.159623

ExtraTrees:
  预测区间: 1.177590 ~ 1.185010 中位预测值: 1.181300

GradientBoosting:
  预测区间: 1.176553 ~ 1.186024 中位预测值: 1.181289

AdaBoost:
  预测区间: 1.171556 ~ 1.184222 中位预测值: 1.177889

KNN:
  预测区间: 1.029000 ~ 1.035400 中位预测值: 1.032200

SVM:
  预测区间: 1.076541 ~ 1.086409 中位预测值: 1.081475



C:\Users\15601\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsRegressor was fitted with feature names
  warnings.warn(
C:\Users\15601\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsRegressor was fitted with feature names
  warnings.warn(
C:\Users\15601\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsRegressor was fitted with feature names
  warnings.warn(
C:\Users\15601\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Pyt